# ML-testbed 15 — coordinate a bounded agent fleet

Lesson 14 assembled one closed-loop synthetic experiment. This lesson turns that loop into **typed work that multiple temporary agents could execute safely**. It models logical work items, a bounded scheduler, creator–validator stages, idempotent delivery, and a controlled knowledge commit.

It does **not** launch LLMs, background workers, or a distributed service. The goal is to inspect the coordination semantics before adding infrastructure.

## How to use this lesson

Run from the top after the earlier ML-testbed lessons, or independently with Python 3.10 or later. No API calls, downloads, model server, PyTorch, or distributed runtime are required.

**Skills:** task graphs, bounded concurrency, typed agent contracts, validation, idempotency, controlled knowledge commits

**Evidence contract:** every result is a deterministic teaching simulation. It demonstrates software invariants; it does not establish multi-agent scale, reliability, cost, or scientific performance in production.

## The architecture under test

```mermaid
flowchart LR
  G[Goal] --> L[Typed task ledger]
  L --> S[Bounded scheduler]
  S --> W[Temporary workers]
  K[Versioned knowledge] --> C[Context compiler]
  C --> W
  W --> V[Validator]
  V -->|accepted| K
  V -->|provisional or rejected| R[Review queue]
  W --> E[Append-only events]
  V --> E
```

The planner proposes dependencies. The scheduler owns admission and concurrency. Workers never write canonical knowledge directly.

In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Callable
import hashlib
import json

@dataclass
class WorkItem:
    task_id: str
    role: str
    objective: str
    depends_on: tuple[str, ...] = ()
    expected_type: str = "Artifact"
    status: str = "planned"
    attempts: int = 0

def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))

def stable_id(prefix, value):
    digest = hashlib.sha256(canonical(value).encode()).hexdigest()[:12]
    return f"{prefix}:{digest}"

## 1. Define a typed evidence-to-decision graph

These are six logical tasks, not six simultaneous model calls. Dependencies determine which work is ready.

In [ ]:
tasks = {
    item.task_id: item for item in [
        WorkItem("task:source-a", "researcher", "Inspect calibration source A"),
        WorkItem("task:source-b", "researcher", "Inspect diagnostic source B"),
        WorkItem("task:claim", "analyst", "Propose an atomic drift claim",
                 ("task:source-a", "task:source-b"), "Claim"),
        WorkItem("task:calculation", "calculator", "Check the bounded interval",
                 ("task:source-a",), "Artifact"),
        WorkItem("task:validate", "validator", "Validate evidence and calculation",
                 ("task:claim", "task:calculation"), "Evaluation"),
        WorkItem("task:decision", "curator", "Commit or withhold the decision",
                 ("task:validate",), "Decision"),
    ]
}

def ready_items(ledger):
    complete = {task_id for task_id, item in ledger.items() if item.status == "accepted"}
    return [item for item in ledger.values()
            if item.status == "planned" and set(item.depends_on) <= complete]

for item in tasks.values():
    deps = ", ".join(item.depends_on) or "—"
    print(f"{item.task_id:<18} {item.role:<11} depends on: {deps}")

## 2. Run bounded scheduling waves

Change `MAX_CONCURRENCY` and rerun. The result remains dependency-correct; only the number of admitted tasks per wave changes.

In [ ]:
MAX_CONCURRENCY = 2
events = []
wave = 0

while any(item.status == "planned" for item in tasks.values()):
    ready = ready_items(tasks)
    if not ready:
        raise RuntimeError("No ready work: the graph is cyclic or a dependency failed")
    admitted = ready[:MAX_CONCURRENCY]
    wave += 1
    print(f"wave {wave}: " + ", ".join(item.task_id for item in admitted))
    for item in admitted:
        item.status = "running"
        item.attempts += 1
        events.append({"event": "leased", "task_id": item.task_id, "attempt": item.attempts})
    for item in admitted:
        item.status = "accepted"
        events.append({"event": "accepted", "task_id": item.task_id, "attempt": item.attempts})

print()
print(f"{len(tasks)} tasks completed in {wave} waves; peak concurrency was bounded at {MAX_CONCURRENCY}.")

## 3. Why dependency graphs replace all-to-all chat

An unrestricted group of `n` agents has `n(n-1)/2` possible pairwise channels. Typed task graphs grow with actual dependencies instead.

In [ ]:
dependency_edges = sum(len(item.depends_on) for item in tasks.values())
print("agents/tasks | pairwise channels | this task graph")
print("-------------|-------------------|----------------")
for n in (10, 100, 1_000, 10_000):
    pairwise = n * (n - 1) // 2
    print(f"{n:>12,} | {pairwise:>17,} | {dependency_edges:>15}")

## 4. Validate before committing knowledge

A worker produces a proposal. A deterministic gate checks its shape and evidence references. The curator—not the worker—owns the canonical write.

In [ ]:
artifacts = {
    "artifact:calibration-a": {"kind": "Source", "verified": True},
    "artifact:diagnostic-b": {"kind": "Observation", "verified": True},
}
proposal = {
    "object_type": "Claim",
    "statement": "Optical-path drift can mimic apparent motion in the synthetic model.",
    "evidence_ids": ["artifact:calibration-a", "artifact:diagnostic-b"],
    "scope": "synthetic-development-only",
    "verification_status": "provisional",
}

def validate_claim(claim, artifact_store):
    errors = []
    required = {"object_type", "statement", "evidence_ids", "scope", "verification_status"}
    if set(claim) != required:
        errors.append("claim shape does not match the contract")
    missing = [ref for ref in claim.get("evidence_ids", []) if ref not in artifact_store]
    if missing:
        errors.append(f"missing evidence: {missing}")
    if claim.get("scope") != "synthetic-development-only":
        errors.append("claim exceeds the evaluated scientific scope")
    return errors

errors = validate_claim(proposal, artifacts)
knowledge = {}
if not errors:
    claim_id = stable_id("claim", proposal)
    knowledge[claim_id] = {**proposal, "verification_status": "accepted"}
    print("controlled commit:", claim_id)
else:
    print("withheld from canonical knowledge:", errors)

## 5. Duplicate delivery must be harmless

Distributed queues may deliver the same task more than once. An idempotency key makes the second commit a replay rather than a second fact.

In [ ]:
commit_log = {}

def commit_once(idempotency_key, value):
    if idempotency_key in commit_log:
        return "replayed", commit_log[idempotency_key]
    object_id = stable_id("artifact", value)
    commit_log[idempotency_key] = object_id
    return "created", object_id

payload = {"task_id": "task:claim", "attempt": 1, "result": proposal}
print(commit_once("task:claim/attempt:1", payload))
print(commit_once("task:claim/attempt:1", payload))
print("canonical records:", len(commit_log))

## What this lab establishes

- Logical task count is separate from model-call concurrency.
- Dependencies make coordination inspectable and prevent premature work.
- Schema and evidence checks occur before knowledge commit.
- Idempotency makes duplicate delivery recoverable.
- The event log preserves attempts; accepted state is a governed projection.

**Try next:** introduce a failed validation, a missing dependency, or a concurrency limit of one. Then decide which events a real Factor runtime must preserve for replay.